# ATML PA2: LLM Post-Training - Kaggle Runner

This notebook provides a clean, modular execution runner for **ATML PA2**.

> **Note on Course Requirements:**
> The assignment forbids submitting a "monolithic notebook" as your pipeline (i.e. putting all training code inside notebook cells with tangled hidden state).
> This runner complies 100% with the requirements: it uses clean, reproducible CLI entry points (`python -m task...`) pointing to our modularized codebase in `https://github.com/moizejaz-31/ATML-PA2`.

---
### Recommended Kaggle Settings:
1. **Accelerator:** GPU T4 x 2 (or GPU P100)
2. **Internet:** **ON** (Required for Hugging Face and course assets)

## 0. Setup Environment & Clone Repository

In [ ]:
# 1. Check GPU
!nvidia-smi

# 2. Clone latest code from GitHub
import os
if not os.path.exists("/kaggle/working/ATML-PA2"):
    !git clone https://github.com/moizejaz-31/ATML-PA2.git /kaggle/working/ATML-PA2
else:
    %cd /kaggle/working/ATML-PA2
    !git pull origin main

%cd /kaggle/working/ATML-PA2

In [ ]:
# 3. Install required pinned packages
!pip install -r requirements.txt -q

In [ ]:
# 4. Download and validate course assets (~1.05 GB)
!python -m scripts.download_assets
!python -m scripts.validate_assets

--- 
## Task 1: Direct Preference Optimization (DPO)
1. **Standard DPO:** Train 1 epoch on UltraFeedback preference data
2. **Evaluate Standard DPO:** Compute held-out preference accuracy, KL drift, RM score, length
3. **Regularization Ablation:** Sweep $\beta \in \{0.03, 0.10, 0.30\}$
4. **Length Confounding:** Train on length-balanced data and test word-limit compliance

In [ ]:
print("=== Task 1.1: Standard DPO Training ===")
!python -m task1_dpo.train --config configs/dpo.yaml --run-name standard

In [ ]:
print("=== Task 1.2: Evaluate Standard DPO ===")
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard

In [ ]:
print("=== Task 1.3: Beta Regularization Ablation ===")
!python -m task1_dpo.ablate_beta --config configs/dpo.yaml

In [ ]:
print("=== Task 1.4: Length Confounding & Word-Limit Study ===")
!python -m task1_dpo.analyze_length --config configs/dpo.yaml

--- 
## Task 2: Proximal Policy Optimization (PPO)
Continues from course-supplied midpoint checkpoints (`checkpoints/ppo_midpoint_policy`, `checkpoints/ppo_midpoint_value`).

In [ ]:
print("=== Task 2.1: Standard PPO Continuation (20 updates) ===")
!python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard

print("=== Task 2.2: Evaluate PPO Policy ===")
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard

print("=== Task 2.3: Clipping Study (epsilon in {0.05, 0.20, 0.50}) ===")
!python -m task2_ppo.analyze_clipping --config configs/ppo.yaml

print("=== Task 2.4: KL-Pressure Study (beta_KL in {0, 0.10, 0.20}) ===")
!python -m task2_ppo.ablate_kl --config configs/ppo.yaml

--- 
## Task 3: Group Relative Policy Optimization (GRPO)
Critic-free RL continuing from `checkpoints/grpo_midpoint_policy`.

In [ ]:
print("=== Task 3.1: Standard GRPO Continuation (20 updates) ===")
!python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard

print("=== Task 3.2: Evaluate GRPO Policy ===")
!python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard

print("=== Task 3.3: Group Size Study (K in {2, 4, 8}) ===")
!python -m task3_grpo.analyze_group_size --config configs/grpo.yaml

print("=== Task 3.4: Normalization Study (Canonical vs Dr. GRPO) ===")
!python -m task3_grpo.compare_normalization --config configs/grpo.yaml

--- 
## Task 5: Feedback Source (RLVR vs RLAIF)
Evaluates frozen course checkpoints (`checkpoints/rlvr_policy` and `checkpoints/rlaif_policy`). Independent of Tasks 1-3.

In [ ]:
print("=== Task 5.1: In-Domain GSM8K Math Evaluation ===")
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset gsm

print("=== Task 5.2: Controlled Reward Diagnostics (100-pair perturbations) ===")
!python -m task5_feedback.score_perturbations --config configs/feedback.yaml

print("=== Task 5.3: Out-of-Domain SVAMP Transfer Evaluation ===")
!python -m task5_feedback.evaluate_math --config configs/feedback.yaml --dataset transfer

print("=== Task 5.4: Compare Feedback Mechanisms ===")
!python -m task5_feedback.compare_feedback --config configs/feedback.yaml

--- 
## Task 4: Safety Calibration
Compares SFT baseline, standard DPO, standard PPO, and standard GRPO policies on XSTest prompts.

In [ ]:
print("=== Task 4.1: Generate Responses for 4 Policies on XSTest ===")
!python -m task4_safety.generate_responses --config configs/feedback.yaml

print("=== Task 4.2: Judge Responses with AI Safety Judge ===")
!python -m task4_safety.judge_responses --config configs/feedback.yaml

print("=== Task 4.3: Generate 60-example Manual Audit Sheet ===")
!python -m task4_safety.make_audit_sheet --config configs/feedback.yaml

print("=== Task 4.4: Safety Calibration Metrics & Disagreements ===")
!python -m task4_safety.evaluate_safety --config configs/feedback.yaml

--- 
## Master Aggregation & Results Packaging

In [ ]:
# Run master visualization to compile all figures and summary tables
!python -m common.visualize_all

# Zip all generated result logs and report figures for 1-click download
!zip -r /kaggle/working/ATML_PA2_ALL_RESULTS.zip results/ report/figures/ outputs/
print("\nSUCCESS! Download 'ATML_PA2_ALL_RESULTS.zip' from the Output tab on the right panel.")